# Data

In [ ]:
import pandas as pd
import re

def clean_arabic_text(text):
    text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
    text = re.sub(r'@\w+|#','', text)

    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "ء", text)
    text = re.sub("ئ", "ء", text)
    text = re.sub("ة", "ه", text)
    text = re.sub("گ", "ك", text)

    arabic_punctuations = r'''`÷×؛<>_()*&^%][ـ،/:"؟.,'{}~+=\"\-'''
    remove_unicode = re.compile(f"[{arabic_punctuations}a-zA-Z0-9]")
    text = re.sub(remove_unicode, " ", text)

    text = re.sub(r'(.)\1+', r'\1\1', text)

    text = text.strip()
    text = " ".join(text.split())
    return text

df = pd.read_csv('/content/40000-Egyptian-tweets.csv')
df['clean_text'] = df['review'].apply(clean_arabic_text)

In [ ]:
!pip install transformers datasets torch -q

In [ ]:
df['labels'] = df['label'].map({'positive': 1, 'negative': 0})

df = df.dropna(subset=['labels'])

df['labels'] = df['labels'].astype('int64')

In [ ]:
df = df[['clean_text','labels']]
df.dropna(inplace=True)

In [ ]:
df

,clean_text,labels
0,اكبر خطا ترتكبه ان تعامل الناس باخلاقك انت مش ...,0
1,داءما اكره اخر ليله في كل مكان .,0
2,يارب اللي يسرق تويتاتي يدخل النار .,0
3,الاسراف في تناول القهوه يسبب الوفاه .,0
4,انا اتبهدلت من التراب النهارده. حاجه تقرف .,1
...,...,...
39995,لنسعد ايامنا بالابتسامه بدلاً ان نملاها بالدموع.,1
39996,مش هقولك غير ان نص الضحك اللي ضحكته في حياتي ك...,1
39997,ربنا يوفقك ويسهلك وان شاء الله تعدي الفتره دي ...,1
39998,مبسوطه اوي عملت طريقه مكرونه جديده و هي دلوقتي...,1


In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.3)

# Model

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "UBC-NLP/MARBERT"
)

In [ ]:
def tokenize(example):
    return tokenizer(
        example["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


In [ ]:
dataset["train"] = dataset["train"].map(tokenize, batched=True)
dataset["test"] = dataset["test"].map(tokenize, batched=True)

Map:   0%|          | 0/27995 [00:00<?, ? examples/s]

Map:   0%|          | 0/11998 [00:00<?, ? examples/s]

In [ ]:
print(dataset["train"].features["labels"])
print(dataset["train"][0]["labels"])
print(type(dataset["train"][0]["labels"]))

Value('float64')
tensor(0.)
<class 'torch.Tensor'>


In [ ]:
dataset["train"].set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
    output_all_columns=False
)

dataset["test"].set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
    output_all_columns=False
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "UBC-NLP/MARBERT",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on yo

In [ ]:
import transformers
print(transformers.__version__)

5.0.0


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    save_strategy="epoch"
)

In [ ]:
from transformers import Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss
500,0.343082
1000,0.297479
1500,0.276029
2000,0.212736
2500,0.163892
3000,0.165947
3500,0.165262
4000,0.075161
4500,0.083135
5000,0.080384


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5250, training_loss=0.18069879368373326, metrics={'train_runtime': 2090.9031, 'train_samples_per_second': 40.167, 'train_steps_per_second': 2.511, 'total_flos': 5524345496102400.0, 'train_loss': 0.18069879368373326, 'epoch': 3.0})

In [ ]:
trainer.save_model("./my_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
trainer.evaluate()

{'eval_loss': 0.4551336467266083,
 'eval_accuracy': 0.9119853308884814,
 'eval_f1': 0.9119844479938112,
 'eval_runtime': 98.8745,
 'eval_samples_per_second': 121.346,
 'eval_steps_per_second': 7.585,
 'epoch': 3.0}

In [ ]:
model.save_pretrained("./my_assistify_model")
tokenizer.save_pretrained("./my_assistify_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_assistify_model/tokenizer_config.json',
 './my_assistify_model/tokenizer.json')

In [ ]:
import torch

In [ ]:
print("Positive" if torch.argmax(model(**tokenizer(
    "يا عم ده ملوش حل",
    padding="max_length", truncation=True, max_length=128, return_tensors="pt"
).to(model.device)).logits, dim=1).item() == 1 else "Negative")

Positive
